이 자료는 위키독스 딥 러닝을 이용한 자연어 처리 입문의 RNN을 이용하여 텍스트 생성하기의 튜토리얼입니다.  

링크 : https://wikidocs.net/45101

2021년 10월 13일에 테스트되었습니다.

# 1. RNN을 이용하여 텍스트 생성하기

In [258]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [259]:
text = """경마장에 있는 말이 뛰고 있다\n
그의 말이 법이다\n
가는 말이 고와야 오는 말이 곱다\n"""

In [260]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
vocab_size = len(tokenizer.word_index) + 1
print('단어 집합의 크기 : %d' % vocab_size)

단어 집합의 크기 : 12


In [261]:
print(tokenizer.word_index)

{'말이': 1, '경마장에': 2, '있는': 3, '뛰고': 4, '있다': 5, '그의': 6, '법이다': 7, '가는': 8, '고와야': 9, '오는': 10, '곱다': 11}


In [262]:
sequences = list()
for line in text.split('\n'): # Wn을 기준으로 문장 토큰화
    encoded = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(encoded)):
        sequence = encoded[:i+1]
        sequences.append(sequence)

print('학습에 사용할 샘플의 개수: %d' % len(sequences))

학습에 사용할 샘플의 개수: 11


In [263]:
print(sequences)

[[2, 3], [2, 3, 1], [2, 3, 1, 4], [2, 3, 1, 4, 5], [6, 1], [6, 1, 7], [8, 1], [8, 1, 9], [8, 1, 9, 10], [8, 1, 9, 10, 1], [8, 1, 9, 10, 1, 11]]


In [264]:
max_len = max(len(l) for l in sequences) # 모든 샘플에서 길이가 가장 긴 샘플의 길이 출력
print('샘플의 최대 길이 : {}'.format(max_len))

샘플의 최대 길이 : 6


In [265]:
sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')

In [266]:
print(sequences)

[[ 0  0  0  0  2  3]
 [ 0  0  0  2  3  1]
 [ 0  0  2  3  1  4]
 [ 0  2  3  1  4  5]
 [ 0  0  0  0  6  1]
 [ 0  0  0  6  1  7]
 [ 0  0  0  0  8  1]
 [ 0  0  0  8  1  9]
 [ 0  0  8  1  9 10]
 [ 0  8  1  9 10  1]
 [ 8  1  9 10  1 11]]


In [267]:
sequences = np.array(sequences)
X = sequences[:,:-1]
y = sequences[:,-1]

In [268]:
print(X)

[[ 0  0  0  0  2]
 [ 0  0  0  2  3]
 [ 0  0  2  3  1]
 [ 0  2  3  1  4]
 [ 0  0  0  0  6]
 [ 0  0  0  6  1]
 [ 0  0  0  0  8]
 [ 0  0  0  8  1]
 [ 0  0  8  1  9]
 [ 0  8  1  9 10]
 [ 8  1  9 10  1]]


In [269]:
print(y) # 모든 샘플에 대한 레이블 출력

[ 3  1  4  5  1  7  1  9 10  1 11]


In [270]:
y = to_categorical(y, num_classes=vocab_size)

In [271]:
print(y)

[[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


In [272]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN

In [273]:
embedding_dim = 10
hidden_units = 32

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(SimpleRNN(hidden_units))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=2)

Epoch 1/200
1/1 - 2s - 2s/step - accuracy: 0.0000e+00 - loss: 2.4889
Epoch 2/200
1/1 - 0s - 44ms/step - accuracy: 0.0909 - loss: 2.4750
Epoch 3/200
1/1 - 0s - 47ms/step - accuracy: 0.0909 - loss: 2.4614
Epoch 4/200
1/1 - 0s - 44ms/step - accuracy: 0.1818 - loss: 2.4478
Epoch 5/200
1/1 - 0s - 42ms/step - accuracy: 0.2727 - loss: 2.4343
Epoch 6/200
1/1 - 0s - 46ms/step - accuracy: 0.3636 - loss: 2.4205
Epoch 7/200
1/1 - 0s - 42ms/step - accuracy: 0.4545 - loss: 2.4063
Epoch 8/200
1/1 - 0s - 42ms/step - accuracy: 0.4545 - loss: 2.3917
Epoch 9/200
1/1 - 0s - 41ms/step - accuracy: 0.4545 - loss: 2.3765
Epoch 10/200
1/1 - 0s - 41ms/step - accuracy: 0.4545 - loss: 2.3606
Epoch 11/200
1/1 - 0s - 43ms/step - accuracy: 0.3636 - loss: 2.3439
Epoch 12/200
1/1 - 0s - 44ms/step - accuracy: 0.3636 - loss: 2.3264
Epoch 13/200
1/1 - 0s - 47ms/step - accuracy: 0.3636 - loss: 2.3080
Epoch 14/200
1/1 - 0s - 44ms/step - accuracy: 0.3636 - loss: 2.2886
Epoch 15/200
1/1 - 0s - 43ms/step - accuracy: 0.3636 - 

In [274]:
def sentence_generation(model, tokenizer, current_word, n): # 모델, 토크나이저, 현재 단어, 반복할 횟수
    init_word = current_word
    sentence = ''

    # n번 반복
    for _ in range(n):
        # 현재 단어에 대한 정수 인코딩과 패딩
        encoded = tokenizer.texts_to_sequences([current_word])[0]
        encoded = pad_sequences([encoded], maxlen=5, padding='pre')
        # 입력한 X(현재 단어)에 대해서 Y를 예측하고 Y(예측한 단어)를 result에 저장.
        result = model.predict(encoded, verbose=0)
        result = np.argmax(result, axis=1)

        for word, index in tokenizer.word_index.items():
            # 만약 예측한 단어와 인덱스와 동일한 단어가 있다면 break
            if index == result:
                break

        # 현재 단어 + ' ' + 예측 단어를 현재 단어로 변경
        current_word = current_word + ' '  + word

        # 예측 단어를 문장에 저장
        sentence = sentence + ' ' + word

    sentence = init_word + sentence
    return sentence

In [275]:
print(sentence_generation(model, tokenizer, '경마장에', 4))

경마장에 있는 말이 뛰고 있다


In [276]:
print(sentence_generation(model, tokenizer, '그의', 2))

그의 말이 법이다


In [277]:
print(sentence_generation(model, tokenizer, '가는', 5))

가는 말이 고와야 오는 말이 곱다


# 2. LSTM을 이용하여 텍스트 생성하기

In [278]:
import pandas as pd
import numpy as np
from string import punctuation

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [279]:
import pandas as pd
import io

# 실제 데이터가 포함된 샘플 CSV 데이터 정의
csv_data = """articleID,articleWordCount,byline,documentType,headline,keywords,multimedia,newDesk,printPage,pubDate
1,150,Jane Doe,Opinion,This is a headline about news,news,image,NYT,1,2018-04-01
2,200,John Smith,News,Another interesting article,politics,video,WP,2,2018-04-02
3,120,Alice Brown,Sports,Breaking news update in sports,sports,text,ESPN,3,2018-04-03
4,180,Bob White,Opinion,Opinion piece on current events,events,audio,BBC,4,2018-04-04
5,210,Charlie Green,News,World events unraveling today,world,image,CNN,5,2018-04-05
"""

df = pd.read_csv(io.StringIO(csv_data))
print("새로운 샘플 데이터프레임의 상위 5행:")
display(df.head())

새로운 샘플 데이터프레임의 상위 5행:


,articleID,articleWordCount,byline,documentType,headline,keywords,multimedia,newDesk,printPage,pubDate
0,1,150,Jane Doe,Opinion,This is a headline about news,news,image,NYT,1,2018-04-01
1,2,200,John Smith,News,Another interesting article,politics,video,WP,2,2018-04-02
2,3,120,Alice Brown,Sports,Breaking news update in sports,sports,text,ESPN,3,2018-04-03
3,4,180,Bob White,Opinion,Opinion piece on current events,events,audio,BBC,4,2018-04-04
4,5,210,Charlie Green,News,World events unraveling today,world,image,CNN,5,2018-04-05


<!-- 이전 샘플 CSV 데이터 설명 - 메인 `df` 로드 셀에 통합되었습니다. -->

In [280]:
print('열의 개수: ',len(df.columns))
print(df.columns)

열의 개수:  10
Index(['articleID', 'articleWordCount', 'byline', 'documentType', 'headline',
       'keywords', 'multimedia', 'newDesk', 'printPage', 'pubDate'],
      dtype='object')


In [281]:
print(df['headline'].isnull().values.any())

False


In [282]:
headline = df['headline'].dropna().tolist() # NaN 값을 제거하고 리스트로 변환
print(f"헤드라인 샘플 5개: {headline[:5]}")

헤드라인 샘플 5개: ['This is a headline about news', 'Another interesting article', 'Breaking news update in sports', 'Opinion piece on current events', 'World events unraveling today']


In [283]:
print('총 샘플의 개수 : {}'.format(len(headline))) # 현재 샘플의 개수

총 샘플의 개수 : 5


In [284]:
 # Unknown 값을 가진 샘플 제거
headline = [word for word in headline if word != "Unknown"]
print('노이즈값 제거 후 샘플의 개수 : {}'.format(len(headline)))

노이즈값 제거 후 샘플의 개수 : 5


In [285]:
# 전처리된 헤드라인으로 토크나이저를 학습시킵니다.
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(preporcessed_headline)
vocab_size = len(tokenizer.word_index) + 1
print('단어 집합의 크기 : %d' % vocab_size)

단어 집합의 크기 : 22


In [286]:
# index_to_word 맵을 올바르게 생성합니다.
index_to_word = {}
for key, value in tokenizer.word_index.items():
    index_to_word[value] = key

print('인덱스-단어 맵의 일부:')
print({k: index_to_word[k] for k in list(index_to_word.keys())[:min(5, len(index_to_word))]})

인덱스-단어 맵의 일부:
{1: 'news', 2: 'events', 3: 'this', 4: 'is', 5: 'a'}


In [287]:
# 시퀀스를 생성합니다.
from tensorflow.keras.preprocessing.sequence import pad_sequences
sequences = list()

for sentence in preporcessed_headline:
    # 각 샘플에 대한 정수 인코딩
    encoded = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(encoded)):
        sequence = encoded[:i+1]
        sequences.append(sequence)

print(f"생성된 시퀀스의 수: {len(sequences)}")
print(sequences[:11])

생성된 시퀀스의 수: 18
[[3, 4], [3, 4, 5], [3, 4, 5, 6], [3, 4, 5, 6, 7], [3, 4, 5, 6, 7, 1], [8, 9], [8, 9, 10], [11, 1], [11, 1, 12], [11, 1, 12, 13], [11, 1, 12, 13, 14]]


In [288]:
from string import punctuation

def repreprocessing(raw_sentence):
    # Ensure raw_sentence is a string before processing
    if not isinstance(raw_sentence, str):
        return "" # Return an empty string or handle as appropriate

    # 구두점 제거와 동시에 소문자화
    preprocessed_sentence = ''.join(word for word in raw_sentence if word not in punctuation).lower()
    return preprocessed_sentence

# repreprocessing 함수를 사용하여 headline 리스트를 전처리합니다.
# headline 리스트는 이미 NaN 값이 제거된 상태이므로 추가적인 필터링이 필요 없습니다.
preporcessed_headline = [repreprocessing(x) for x in headline]
print(f"전처리된 헤드라인 샘플 5개: {preporcessed_headline[:5]}")

전처리된 헤드라인 샘플 5개: ['this is a headline about news', 'another interesting article', 'breaking news update in sports', 'opinion piece on current events', 'world events unraveling today']


In [289]:
max_len = max(len(l) for l in sequences)
print('샘플의 최대 길이 : {}'.format(max_len))

샘플의 최대 길이 : 6


In [290]:
sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
print(sequences[:3])

[[0 0 0 0 3 4]
 [0 0 0 3 4 5]
 [0 0 3 4 5 6]]


In [291]:
sequences = np.array(sequences)
X = sequences[:,:-1]
y = sequences[:,-1]

In [292]:
print(X[:3])

[[0 0 0 0 3]
 [0 0 0 3 4]
 [0 0 3 4 5]]


In [293]:
print(y[:3])

[4 5 6]


In [294]:
y = to_categorical(y, num_classes=vocab_size)

In [295]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM

In [296]:
embedding_dim = 10
hidden_units = 128

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(LSTM(hidden_units))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=2)

Epoch 1/200
1/1 - 1s - 1s/step - accuracy: 0.0000e+00 - loss: 3.0913
Epoch 2/200
1/1 - 0s - 45ms/step - accuracy: 0.0556 - loss: 3.0883
Epoch 3/200
1/1 - 0s - 44ms/step - accuracy: 0.1111 - loss: 3.0853
Epoch 4/200
1/1 - 0s - 43ms/step - accuracy: 0.1111 - loss: 3.0822
Epoch 5/200
1/1 - 0s - 48ms/step - accuracy: 0.1111 - loss: 3.0790
Epoch 6/200
1/1 - 0s - 46ms/step - accuracy: 0.1111 - loss: 3.0755
Epoch 7/200
1/1 - 0s - 47ms/step - accuracy: 0.1111 - loss: 3.0717
Epoch 8/200
1/1 - 0s - 45ms/step - accuracy: 0.1111 - loss: 3.0675
Epoch 9/200
1/1 - 0s - 44ms/step - accuracy: 0.1111 - loss: 3.0629
Epoch 10/200
1/1 - 0s - 53ms/step - accuracy: 0.1111 - loss: 3.0577
Epoch 11/200
1/1 - 0s - 50ms/step - accuracy: 0.1111 - loss: 3.0520
Epoch 12/200
1/1 - 0s - 45ms/step - accuracy: 0.1111 - loss: 3.0455
Epoch 13/200
1/1 - 0s - 46ms/step - accuracy: 0.1111 - loss: 3.0382
Epoch 14/200
1/1 - 0s - 45ms/step - accuracy: 0.1111 - loss: 3.0300
Epoch 15/200
1/1 - 0s - 63ms/step - accuracy: 0.1111 - 

In [297]:
def sentence_generation(model, tokenizer, current_word, n): # 모델, 토크나이저, 현재 단어, 반복할 횟수
    init_word = current_word
    sentence = ''

    # n번 반복
    for _ in range(n):
        encoded = tokenizer.texts_to_sequences([current_word])[0]
        encoded = pad_sequences([encoded], maxlen=max_len-1, padding='pre')

        # 입력한 X(현재 단어)에 대해서 y를 예측하고 y(예측한 단어)를 result에 저장.
        result = model.predict(encoded, verbose=0)
        result = np.argmax(result, axis=1)

        for word, index in tokenizer.word_index.items():
            # 만약 예측한 단어와 인덱스와 동일한 단어가 있다면
            if index == result:
                break

        # 현재 단어 + ' ' + 예측 단어를 현재 단어로 변경
        current_word = current_word + ' '  + word

        # 예측 단어를 문장에 저장
        sentence = sentence + ' ' + word

    sentence = init_word + sentence
    return sentence

In [298]:
print(sentence_generation(model, tokenizer, 'i', 10))

i piece piece events unraveling today about news news news today


In [299]:
print(sentence_generation(model, tokenizer, 'how', 10))

how piece piece events unraveling today about news news news today
